# Extract DEX prices from Dune

Pull swap-level mid prices for one token pair on one blockchain, from each supported DEX, over a **collection window**, then save one CSV per DEX under `<chain>/`.

All reusable logic lives in the `arblib` package; this notebook only sets parameters and wires the steps together.

In [1]:
# !pip install -r requirements.txt

In [1]:
from arblib.config import SWAP_QUERY_IDS, LIQUIDITY_QUERY_IDS, GAS_QUERY_IDS, TOKENS, build_collection_params
from arblib.dune_api import make_headers, run_dune_saved_query
from arblib.data_io import save_dataframes
import os
from pathlib import Path


## Parameters

- **CHAIN / tokens** — which market to pull.
- **Collection window** (`START_TS` / `END_TS`, UTC) — note `START_TS` is deliberately *earlier* than the study start used in `arbitrage.ipynb`, so every pool already has a known price to forward-fill from once the study window begins.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  
DUNE_API_KEY = os.environ["DUNE_API_KEY"]

current = Path.cwd()
while current.name != 'defi_arbitrage' and current != current.parent:
    current = current.parent
BASE_DIR = current if current.name == 'defi_arbitrage' else Path.cwd()

# --- What to collect ------------------------------------------------
CHAIN  = "ethereum"                 # blockchain name
TOKEN0 = TOKENS[CHAIN]["WETH"]  # base token
TOKEN1 = TOKENS[CHAIN]["USDC"]  # quote token

# --- Collection window (UTC) ---------------------------------------
START_TS = "2025-12-31 15:00:00"
END_TS   = "2025-12-31 16:00:00"




params  = build_collection_params(CHAIN, TOKEN0, TOKEN1, START_TS, END_TS)
headers = make_headers(DUNE_API_KEY)
params

{'start_ts': '2025-12-31 15:00:00',
 'end_ts': '2025-12-31 16:00:00',
 'token0': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'token1': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
 'chain': 'ethereum'}

## Run the swap queries

Swap-level mid prices, one query per DEX, then **left-joined with that DEX's per-swap gas** (on block / pool / tx hash / event index) so the gas cost rides along with each swap. A DEX not available on `CHAIN` returns an empty DataFrame and is skipped on save. The merged frames are saved under `<chain>/swaps/`.

In [3]:
merge_keys = ["evt_block_number", "pool", "evt_tx_hash", "evt_index"]

# Pancake: swap-level mid prices + per-swap gas, left-joined onto the swaps so
# the gas cost rides along with each swap (saved under swaps/, not gas/).
df_pancake_swap = run_dune_saved_query(SWAP_QUERY_IDS["pancake"], params, headers, "Pancake")
df_gas_pancake  = run_dune_saved_query(GAS_QUERY_IDS["pancake_gas_per_swap"], params, headers, "Gas pancake")
if not df_pancake_swap.empty and not df_gas_pancake.empty:
    df_pancake_swap = df_pancake_swap.merge(df_gas_pancake, on=merge_keys, how="left")

# Uniswap: same.
df_uniswap_swap = run_dune_saved_query(SWAP_QUERY_IDS["uniswap"], params, headers, "Uniswap")
df_gas_uniswap  = run_dune_saved_query(GAS_QUERY_IDS["uniswap_gas_per_swap"], params, headers, "Gas uniswap")
if not df_uniswap_swap.empty and not df_gas_uniswap.empty:
    df_uniswap_swap = df_uniswap_swap.merge(df_gas_uniswap, on=merge_keys, how="left")


[Pancake] EXECUTE RESPONSE: {'execution_id': '01KVDWC6A4N9T6A7PC3S0HP24N', 'state': 'QUERY_STATE_PENDING'}
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_COMPLETED
[Gas pancake] EXECUTE RESPONSE: {'execution_id': '01KVDWCFQ3TAAR98Y4QXAMPEDD', 'state': 'QUERY_STATE_PENDING'}
[Gas pancake] STATUS: QUERY_STATE_PENDING
[Gas pancake] STATUS: QUERY_STATE_COMPLETED
[Uniswap] EXECUTE RESPONSE: {'execution_id': '01KVDWCRZQKS87811TXGTFXGKB', 'state': 'QUERY_STATE_PENDING'}
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_COMPLETED
[Gas uniswap] EXECUTE RESPONSE: {'execution_id': '01KVDWD7JNJMX5F3FDA3J98VVF', 'state': 'QUERY_STATE_PENDING'}
[Gas uniswap] STATUS: QUERY_STATE_PENDING
[Gas uniswap] STATUS: QUERY_STATE_COMPLETED


In [4]:
swap_dir = os.path.join(BASE_DIR, CHAIN, "swaps")

save_dataframes(
    {
        "df_uniswap_swap.csv": df_uniswap_swap,
        "df_pancake_swap.csv": df_pancake_swap,
    },
    swap_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/swaps/df_uniswap_swap.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/swaps/df_pancake_swap.csv
Done.


## Run the liquidity queries

Mint / burn events per pool, one query per DEX, over the same `params` window. Used downstream to reconstruct the liquidity state at any block.

In [6]:
df_uniswap_liq = run_dune_saved_query(LIQUIDITY_QUERY_IDS["uniswap"], params, headers, "Uniswap liquidity")
df_pancake_liq = run_dune_saved_query(LIQUIDITY_QUERY_IDS["pancake"], params, headers, "Pancake liquidity")


[Uniswap liquidity] EXECUTE RESPONSE: {'execution_id': '01KVDNNR6S7GZ2HBT2JNVSGTQA', 'state': 'QUERY_STATE_PENDING'}
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_COMPLETED
[Pancake liquidity] EXECUTE RESPONSE: {'execution_id': '01KVDNQD18PE277PSKGYYBAS48', 'state': 'QUERY_STATE_PENDING'}
[Pancake liquidity] STATUS: QUERY_STATE_PENDING
[Pancake liquidity] STATUS: QUERY_STATE_EXECUTING
[Pancake liquidity] STATUS: QUERY_STATE_COMPLETED
[INFO] Pancake liq

In [7]:
liquidity_dir = os.path.join(BASE_DIR, CHAIN, "liquidity")

save_dataframes(
    {
        "df_uniswap_liq.csv": df_uniswap_liq,
        "df_pancake_liq.csv": df_pancake_liq,
    },
    liquidity_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/liquidity/df_uniswap_liq.csv
Skipped empty or missing dataframe: df_pancake_liq.csv
Done.


## Run the gas query

Per-block base fee + utilization over the same collection window, for the whole chain (not per-DEX). Used downstream to price the gas cost of an arbitrage at any block. The **per-swap** gas is collected with the swaps above and saved under `swaps/`; only this chain-wide gas is saved under `gas/`.

In [9]:
df_gas_chain = run_dune_saved_query(GAS_QUERY_IDS["chain_gas_price"], params, headers, "Gas price")

[Gas price] dropping params not used by query 7748900: ['token0', 'token1']
[Gas price] EXECUTE RESPONSE: {'execution_id': '01KVDNQT3EE6RA56DAZVACHV7Z', 'state': 'QUERY_STATE_PENDING'}
[Gas price] STATUS: QUERY_STATE_EXECUTING
[Gas price] STATUS: QUERY_STATE_COMPLETED


In [ ]:
gas_dir = os.path.join(BASE_DIR, CHAIN, "gas")

save_dataframes(
    {
        "chain_gas_price.csv": df_gas_chain,
    },
    gas_dir,
)